In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import glob
import pandas as pd
from torch.utils.data import Dataset, DataLoader,random_split
import seaborn as sns
import timm
import torch.optim as optim
import time
from PIL import Image
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

import warnings
warnings.filterwarnings('ignore')

### Classes and dataset configuration

In [ ]:
#Features ordenation
dataset_dict = {
    'Dataset1': ["Bel-Pao","Copia","ERV","Gypsy","L1","Mariner","Non-TE","hAT"],
    'Dataset2': ["LINE","LTR","Non-TE","CLASS II"],
    'Dataset3': ["Bel-Pao","CACTA","Copia","ERV","Gypsy","L1","Mariner","Mutator","PIF","Non-TE","SINE","hAT"],
    'Dataset4': ["LINE","LTR","Non-TE", "SINE","CLASS II"],
    'Dataset5': ["LINE","LTR","Non-TE","CLASS II"]
}

# Desired order
dataset_dict_artigo = {
    'Dataset1': ["Copia","Gypsy","Bel-Pao","ERV","L1","Mariner","hAT","Non-TE"],
    'Dataset2': ["LTR","LINE","CLASS II","Non-TE"],
    'Dataset3': ["Copia","Gypsy","Bel-Pao","ERV","L1","SINE","Mariner","hAT","Mutator","PIF","CACTA","Non-TE"],
    'Dataset4': ["LTR","LINE", "SINE","CLASS II","Non-TE"],
    'Dataset5': ["LTR","LINE","CLASS II","Non-TE"]
}
# Mapping for datasets
dataset_dict_ord_artigo = {
    'Dataset1': [1,3,0,2,4,5,7,6],
    'Dataset2': [1,0,3,2],
    'Dataset3': [2,4,0,3,5,10,6,11,7,8,1,9],
    'Dataset4': [1,0, 3,4,2],
    'Dataset5': [1,0,3,2]
}

### Configure dataset to process

In [ ]:
dataset_processing =  'Dataset1' # Change this to select dataset
labels = dataset_dict[dataset_processing]
print(labels)
class_names = pd.factorize(labels)[0]
print(class_names)
correct_labels = []
list_image = []
temp = '../generated-images/' # Path to generated images
path_imagens = temp+dataset_processing+'/'
for cls in class_names:

    class_correct = []
    caminho = path_imagens + 'Train/classe_' + str(cls) + '/*.png'
    list_image_path = glob.glob(caminho,recursive=True)
    for img in list_image_path:
        correct_labels.append(labels[cls])
    list_image.extend(list_image_path)


### Prepare the dataset

In [ ]:

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.0001
BATCH_SIZE = 32
NUM_EPOCHS = 100
IMAGE_SIZE = 224
PATCH_SIZE = 8
NUM_PATCHES = (IMAGE_SIZE // PATCH_SIZE) ** 2
PROJECTION_DIM = 64
NUM_HEADS = 4
TRANSFORMER_LAYERS = 8
MLP_HEAD_UNITS = [2048, 1024]

# Define image transformations
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.repeat(3, 1, 1)) # Repeat grayscale to 3 channels
])

# Define a custom dataset
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, list_image_path,list_txt):
        self.image_path = list_image_path
        self.labels  = pd.factorize(list_txt)[0]

    def __len__(self):
        return len(self.image_path)

    def __getitem__(self, idx):
        image = transform(Image.open(self.image_path[idx]).convert('L'))
        title = self.labels[idx]
        title = torch.tensor(title, dtype=torch.long)
        return image, title


dataset = CustomDataset(list_image, correct_labels)

# Define train and validation proportions
train_size = int(0.8 * len(dataset))  # 80% for training
valid_size = len(dataset) - train_size  # 20% for validation

# Split the dataset into training and validation
train_dataset, valid_dataset = random_split(dataset, [train_size, valid_size])

# Create DataLoaders for training and validation
trainloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
validloader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)


correct_labels_test = []
list_image_test = []
for cls in class_names:

    class_correct = []
    caminho = path_imagens + 'Test/classe_' + str(cls) + '/*.png'
    list_image_test_path = glob.glob(caminho,recursive=True)
    for img in list_image_test_path:
        correct_labels_test.append(labels[cls])
    list_image_test.extend(list_image_test_path)
test_dataset = CustomDataset(list_image_test,correct_labels_test)
testloader = DataLoader(test_dataset, shuffle=False, batch_size=BATCH_SIZE, generator=torch.Generator(device=DEVICE))



### Model

In [ ]:
model = timm.create_model('resnetrs50', pretrained=True)
num_features = model.num_features
model.fc = nn.Sequential(
    nn.Flatten(),
    nn.Linear(num_features, 512),
    nn.ReLU(),
    nn.Linear(512, len(class_names)),
    nn.Softmax(dim=1)
)
device = DEVICE
model.to(DEVICE)

### Train

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.0001)

num_epochs = 100
best_accuracy = 0.0
start = time.time()
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in validloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(trainloader):.4f}, Accuracy: {accuracy:.2f}%")
    # Save the best model based on accuracy
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        torch.save(model.state_dict(), 'resnet-100-'+dataset_processing+'.pth')
end = time.time()
print("Fineshed training!")
print("Time: ", end - start)

### Evaluate on test set

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generator = torch.Generator(device="cpu")
testloader = torch.utils.data.DataLoader(test_dataset, batch_size=32, generator=generator)

# Load the saved model

model.load_state_dict(torch.load('resnet-100-'+dataset_processing+'.pth'))
model.to(device)
model.eval()

# Lists to store true labels and predictions
all_preds = []
all_labels = []

# Disable gradient calculation for memory efficiency
with torch.no_grad():
    for images, labels in testloader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

### Confusion matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

labels = dataset_dict[dataset_processing]

# Define the new order of labels and reorder the confusion matrix
new_order = dataset_dict_ord_artigo[dataset_processing]  # Desired order of classes
cm_reordered = cm[new_order, :]  # Reorder rows
cm_reordered = cm_reordered[:, new_order]  # Reorder columns
disp = ConfusionMatrixDisplay(confusion_matrix=cm_reordered, display_labels=dataset_dict_artigo[dataset_processing])
fig, ax = plt.subplots(figsize=(10,10))
disp.plot(ax=ax,cmap = 'Blues')
plt.xticks(rotation=45)
plt.yticks(rotation=45)
plt.xlabel("Predict")
plt.ylabel("True")
plt.show()


### Print classification report

In [ ]:
metrics = classification_report(all_labels, all_preds, target_names=labels, output_dict=True)
metrics_df = pd.DataFrame(metrics).transpose()
metrics_df